# Day 8 — Solution: Covariance & Correlation

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
import os
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

## E1 — by hand

x̄ = 6, ȳ = 3; dx=(−4,−2,0,2,4), dy=(−2,0,−1,2,1);
Σdx·dy = 8+0+0+4+4 = 16 → Cov = 16/5 = 3.2 (population convention; /4 if
sample). ρ = 3.2/(2·1.41) ≈ 0.68 — moderately positive: y tends above its
mean when x is.

In [ ]:
x = np.array([2, 4, 6, 8, 10.]); y = np.array([1, 3, 2, 5, 4.])
print(np.cov(x, y, ddof=0)[0, 1], np.corrcoef(x, y)[0, 1])

## E2 — correlation noise

In [ ]:
rng = np.random.default_rng(42)
for T in [63, 252]:
    rhos = []
    for _ in range(2000):
        a, b = rng.normal(size=T), rng.normal(size=T)
        rhos.append(np.corrcoef(a, b)[0, 1])
    rhos = np.array(rhos)
    print(f"T={T}: mean {rhos.mean():+.3f} sd {rhos.std():.3f} "
          f"(theory {1/np.sqrt(T):.3f}) | P(|ρ̂|>0.25) = {(abs(rhos) > 0.25).mean():.1%}")

**T=63: SD ≈ 0.126 (theory 1/√63 = 0.126 ✓); P(|ρ̂|>0.25) ≈ 5%** (it is
the 2-SE tail), but P(|ρ̂|>0.126) ≈ 33% — one SE is crossed a third of
the time. T=252: SD ≈ 0.063, P(|ρ̂|>0.25) ≈ 0.01%. A quarter-length
"correlation" of truly independent series routinely reaches ±0.13–0.2 —
**quarterly correlation analysis is noise sculpture; only |ρ̂| beyond
~0.25 even clears the 2-SE bar, and even that is 1-in-20 luck.**

## E3 — real correlations with error bars

In [ ]:
if DATA_SOURCE == "real":
    tickers = ["SPY", "TLT", "GLD", "XLE", "QQQ"]
    px = get_prices(tickers, start="2010-01-01")
else:
    tickers = ["S0", "S1", "S2", "S3", "S4"]
    px = synthetic_prices(n_days=3000, n_assets=5, seed=18, corr=0.2)
    px.columns = tickers
r = px.pct_change().dropna()
T = len(r)
C = r.corr()
print(C.round(2))
for i in range(len(tickers)):
    for j in range(i + 1, len(tickers)):
        rho = C.iloc[i, j]
        se = (1 - rho ** 2) / np.sqrt(T)
        print(f"{tickers[i]}-{tickers[j]}: ρ={rho:+.2f} ± {1.96*se:.2f} "
              f"{'(≠0)' if abs(rho) > 2*se else '(≈0)'}")

With T≈3,700, SE ≈ 0.015–0.02: nearly all real pairs clear the bar —
long samples resolve correlations well. The interesting question is never
"is it nonzero" but "is it stable" — E4.

## E4 — rolling ρ(SPY, TLT)

In [ ]:
if DATA_SOURCE == "real":
    pair = r[["SPY", "TLT"]]
else:
    pair = r[[tickers[0], tickers[1]]]
roll = pair.iloc[:, 0].rolling(252).corr(pair.iloc[:, 1]).dropna()
full = pair.iloc[:, 0].corr(pair.iloc[:, 1])
print(f"full-sample {full:+.2f} | rolling min {roll.min():+.2f} max {roll.max():+.2f} "
      f"| sign differs {np.mean(np.sign(roll) != np.sign(full)):.0%} of the time")
roll.plot(); plt.axhline(full, color="red", ls="--"); plt.show()

Real SPY/TLT: full-sample ≈ −0.3, but the rolling ρ wanders from ≈ −0.6
to ≈ +0.4, flipping sign a nontrivial fraction of the time. **"ρ = −0.3"
conceals: (1) the range, (2) the regime-conditional structure, (3) that
the average is a mixture, not a property.** A hedge sized on the average
is mis-sized in every regime except the average one.

## E5 — false conclusion factory (exemplar)

"We're diversified: 60% equity index, 40% commodity fund, historical ρ =
0.15." In the 2008-style stress, both legs respond to the same liquidity
shock — correlations converge toward 1 exactly as both fall. The variance
formula Var = Σwᵢwⱼσᵢσⱼρᵢⱼ: ρ jumping 0.15→0.8 raises the cross term
~5×, and the portfolio's diversification benefit — the whole reason for
the 40% — evaporates at the moment of maximum pain. Correlation is a
regime-dependent quantity; its stress value, not its average, sets your
worst day (Part 3 of day 14 measures exactly this).